# 8. Question 4: what does erasing race information do to the errors?

**Least-squares concept erasure (LEACE)** changes the embeddings minimally so that no linear classifier can predict the group (notebook 00, section 0.8). Within each round of the cross-fitting it is estimated from the standardised fitting ECGs of the four groups and applied to every ECG; the detectors of notebook 07 are then **refitted** on the erased embeddings (C from 0.001, 0.01, 0.1 by validation AUROC; 80%-sensitivity threshold) and applied to the evaluation patients.

Because this **standard erasure** also removes the differences that exist only because a disease is more common in one group, LEACE is also estimated from fitting ECGs **without the diagnosis in question (disease-negative erasure)**: without composite SHD for the composite detector, and without aortic stenosis for the aortic-stenosis detector.

New race probes refitted on the erased embeddings check the removal (Black-versus-White AUROC, mean of the five rounds).

Solvers (notebook 00, section 0.5): EchoNext standard and SHD-negative erasure with scikit-learn on the CPU (parallel workers); EchoNext aortic-stenosis-negative erasure and all MIMIC-IV erasure with the GPU solver.

**Paper items:** Figs. 3 and 4, Supplementary Table 7, the "After LEACE" column of Supplementary Table 3, Table 2 (questions 6 and 7).

Run time: about 11 minutes for EchoNext (10 CPU workers), 2 minutes for MIMIC-IV and 1 minute for the bootstrap.

In [ ]:
%run ./00_settings.ipynb
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

OUT = results("08_race_erasure")
STD, SHD_NEG, AS_NEG = "LEACE", "LEACE (SHD-neg fit)", "LEACE (AS-neg fit)"

## 8.1 EchoNext: standard and SHD-negative erasure (scikit-learn, CPU)

In [ ]:
D = load_design("echonext")
rows = [fold_rows(D, k) for k in range(N_FOLDS)]
g_row = D["g"][D["pi_of_row"]]
n_pat = len(D["fold_of"])
folder = work("echonext", "erasure")


def race_bw_auc_cpu(X, g):
    """Black-versus-White AUROC on the evaluation patients of a probe refitted on (erased) features (C from 0.001-1 by validation AUROC)."""
    bw = {n: np.isin(g[n], [B, W]) for n in X}
    _, _, wb, bb = lr_select_cpu(X["tr"][bw["tr"]], (g["tr"][bw["tr"]] == B).astype(int), X["va"][bw["va"]], (g["va"][bw["va"]] == B).astype(int), C_ERASED_RACE)
    s = X["ev"][bw["ev"]] @ wb.ravel() + bb[0]
    return float(roc_auc_score((g["ev"][bw["ev"]] == B).astype(int), s))


def erasure_worker(fm, k, rows_k, Y_row, g_row):
    with threadpool_limits(2):
        X = emb_echonext(fm)
        idx = dict(zip(("tr", "va", "ev"), rows_k))
        sc = StandardScaler().fit(X[idx["tr"]].astype(np.float64))
        Xs = {n: sc.transform(X[i].astype(np.float64)) for n, i in idx.items()}
        del X
        g = {n: g_row[i] for n, i in idx.items()}
        Y = {n: Y_row[i] for n, i in idx.items()}
        ktr = g["tr"] >= 0                                                    # erasers are fitted on the four groups
        neg = ktr & (Y["tr"][:, 0] == 0)                                      # ... and on fitting ECGs without composite SHD
        er = {STD: leace_fit_np(Xs["tr"][ktr], onehot(g["tr"][ktr]))[:2], SHD_NEG: leace_fit_np(Xs["tr"][neg], onehot(g["tr"][neg]))[:2]}
        out = {}
        for name, (mu, M) in er.items():
            Xe = {n: v - (v - mu) @ M.T for n, v in Xs.items()}
            res = {"p": {}, "thr": {}}
            if name == STD:
                res["race_bw"] = race_bw_auc_cpu(Xe, g)
            for j in (range(Y_row.shape[1]) if name == STD else [0]):          # standard: every diagnosis; SHD-negative: composite SHD
                _, _, w, b = lr_select_cpu(Xe["tr"], Y["tr"][:, j], Xe["va"], Y["va"][:, j], C_ERASED_DETECTOR)
                pv, pe = sigmoid(Xe["va"] @ w.ravel() + b[0]), sigmoid(Xe["ev"] @ w.ravel() + b[0])
                res["thr"][j] = float(np.quantile(pv[Y["va"][:, j] == 1], 1 - SENS))
                res["p"][j] = pe
            out[name] = res
        return fm, k, out


todo = [fm for fm in FMS if not (folder / f"erased_{fm}.npz").exists()]
if todo:
    t0 = time.time()
    outs = Parallel(n_jobs=CPU_JOBS, backend="loky")(delayed(erasure_worker)(fm, k, rows[k], D["Y_row"], g_row) for fm in todo for k in range(N_FOLDS))
    for fm in todo:
        fo = {k: o for f, k, o in outs if f == fm}
        save = {}
        for name in (STD, SHD_NEG):
            P1 = np.full((n_pat, len(E_LABELS)), np.nan); PR1 = np.zeros((n_pat, len(E_LABELS)), np.int8)
            for k in range(N_FOLDS):
                pe = D["pi_of_row"][rows[k][2]]
                for j, p in fo[k][name]["p"].items():
                    P1[pe, j], PR1[pe, j] = p, (p >= fo[k][name]["thr"][j]).astype(np.int8)
            save[f"p__{name}"], save[f"pred__{name}"] = P1.astype(np.float32), PR1
        save["race_bw"] = np.array([fo[k][STD]["race_bw"] for k in range(N_FOLDS)])
        np.savez_compressed(folder / f"erased_{fm}.npz", **save)
    print(f"EchoNext standard and SHD-negative erasure: {(time.time() - t0) / 60:.1f} min")

## 8.2 EchoNext: aortic-stenosis-negative erasure (GPU solver)

In [ ]:
j_as = E_LABELS.index(E_AS)
for fm in FMS:
    f = folder / f"erased_as_neg_{fm}.npz"
    if f.exists():
        continue
    t0 = time.time()
    X = emb_echonext(fm)
    P1, PR1 = np.full(n_pat, np.nan), np.zeros(n_pat, np.int8)
    for k, (tr, va, ev) in enumerate(rows):
        idx = {"tr": tr, "va": va, "ev": ev}
        Xs = {n: to_dev(X[i]) for n, i in idx.items()}
        standardize_(Xs["tr"], Xs["va"], Xs["ev"])
        gg, yy = {n: g_row[i] for n, i in idx.items()}, {n: D["Y_row"][i, j_as] for n, i in idx.items()}
        neg = (gg["tr"] >= 0) & (yy["tr"] == 0)                               # fitting ECGs of the four groups without aortic stenosis
        er = leace_fit_torch(Xs["tr"][torch.as_tensor(neg, device=device())], gg["tr"][neg])[:2]
        Xe = {n: leace_apply_torch(v, er) for n, v in Xs.items()}
        b = lr_select(Xe["tr"], yy["tr"].astype(int), Xe["va"], yy["va"].astype(int), C_ERASED_DETECTOR)
        thr = float(np.quantile(sigmoid(lr_scores(Xe["va"], b["W"], b["b"]))[yy["va"] == 1], 1 - SENS))
        p = sigmoid(lr_scores(Xe["ev"], b["W"], b["b"]))
        pe = D["pi_of_row"][ev]
        P1[pe], PR1[pe] = p, (p >= thr).astype(np.int8)
        del Xs, Xe
        free_gpu()
    np.savez_compressed(f, p=P1.astype(np.float32), pred=PR1)
    print(f"{fm}: {time.time() - t0:.0f} s")

## 8.3 MIMIC-IV: standard, SHD-negative and aortic-stenosis-negative erasure (GPU solver)

In [ ]:
C = load_mimic_cohort()
Dm = load_design("mimic")
M_LABELS = mimic_labels()
Yrow = C[M_LABELS].to_numpy(float)
grow = race_index(C.race6)
rows_m = C.row.to_numpy()
n_pat_m = len(Dm["fold_of"])
folder_m = work("mimic", "erasure")
j_as_m = M_LABELS.index(M_AS)


def race_bw_auc_gpu(X, g):
    bw = {n: np.isin(g[n], [B, W]) for n in X}
    bb = lr_select(X["tr"][torch.as_tensor(bw["tr"], device=device())], (g["tr"][bw["tr"]] == B).astype(int),
                   X["va"][torch.as_tensor(bw["va"], device=device())], (g["va"][bw["va"]] == B).astype(int), C_ERASED_RACE)
    s = lr_scores(X["ev"][torch.as_tensor(bw["ev"], device=device())], bb["W"], bb["b"])
    return fast_auc(g["ev"][bw["ev"]] == B, s)


def refit(Xe, y, ev_pat, P1, PR1):
    """Detector refitted on erased features; fills the evaluation patients' probabilities and predictions."""
    b = lr_select(Xe["tr"], y["tr"].astype(int), Xe["va"], y["va"].astype(int), C_ERASED_DETECTOR)
    thr = float(np.quantile(sigmoid(lr_scores(Xe["va"], b["W"], b["b"]))[y["va"] == 1], 1 - SENS))
    p = sigmoid(lr_scores(Xe["ev"], b["W"], b["b"]))
    P1[ev_pat], PR1[ev_pat] = p, (p >= thr).astype(np.int8)


for fm in FMS:
    f = folder_m / f"erased_{fm}.npz"
    if f.exists():
        continue
    t0 = time.time()
    X = emb_mimic(fm)[rows_m]
    P1 = {e: np.full((n_pat_m, len(M_LABELS)), np.nan) for e in (STD, SHD_NEG, AS_NEG)}
    PR1 = {e: np.zeros((n_pat_m, len(M_LABELS)), np.int8) for e in (STD, SHD_NEG, AS_NEG)}
    race_bw = []
    for k in range(N_FOLDS):
        idx = dict(zip(("tr", "va", "ev"), fold_rows(Dm, k)))
        Xs = {n: to_dev(X[i]) for n, i in idx.items()}
        standardize_(Xs["tr"], Xs["va"], Xs["ev"])
        g = {n: grow[i] for n, i in idx.items()}
        Y = {n: Yrow[i] for n, i in idx.items()}
        ktr = g["tr"] >= 0
        masks = {STD: ktr, SHD_NEG: ktr & (Y["tr"][:, 0] == 0), AS_NEG: ktr & (Y["tr"][:, j_as_m] == 0)}
        er = {e: leace_fit_torch(Xs["tr"][torch.as_tensor(m, device=device())], g["tr"][m])[:2] for e, m in masks.items()}
        pe = Dm["pi_of_row"][idx["ev"]]
        for name in (STD, SHD_NEG, AS_NEG):
            Xe = {n: leace_apply_torch(v, er[name]) for n, v in Xs.items()}
            if name == STD:
                race_bw.append(race_bw_auc_gpu(Xe, g))
            for j in {STD: range(len(M_LABELS)), SHD_NEG: [0], AS_NEG: [j_as_m]}[name]:
                P1j, PR1j = P1[name][:, j].copy(), PR1[name][:, j].copy()
                refit(Xe, {n: Y[n][:, j] for n in Y}, pe, P1j, PR1j)
                P1[name][:, j], PR1[name][:, j] = P1j, PR1j
            del Xe
            free_gpu()
        del Xs, er
        free_gpu()
    np.savez_compressed(f, **{f"p__{e}": P1[e].astype(np.float32) for e in P1}, **{f"pred__{e}": PR1[e] for e in PR1}, race_bw=np.array(race_bw))
    print(f"{fm}: {(time.time() - t0) / 60:.1f} min")

## 8.4 Race decodability after erasure, and the detectors' AUROC before and after (Supplementary Tables 3 and 7)

In [ ]:
lat_m = C.loc[Dm["lat_idx"]].reset_index(drop=True)
Y_lat = {"EchoNext": D["Y"], "MIMIC-IV": lat_m[M_LABELS].to_numpy(float)}
race, auc_rows = [], []
for cohort, fold, labels, base_folder in (("EchoNext", folder, E_LABELS, work("echonext", "detectors")), ("MIMIC-IV", folder_m, M_LABELS, work("mimic", "detectors"))):
    for fm in FMS:
        z = np.load(fold / f"erased_{fm}.npz")
        race.append({"cohort": cohort, "fm": FMS[fm], "fm_key": fm, "eraser": STD, "race_bw_auroc": float(z["race_bw"].mean()),
                     "race_bw_min": float(z["race_bw"].min()), "race_bw_max": float(z["race_bw"].max())})
        base = np.load(base_folder / f"oof_{fm}.npz")
        for j, lab in enumerate(labels):
            y = Y_lat[cohort][:, j]
            auc_rows.append({"cohort": cohort, "fm": FMS[fm], "fm_key": fm, "label_key": lab, "label": LABEL_NAME[lab], "eraser": STD,
                             "auroc_before": fast_auc(y, base["p"][:, j].astype(float)), "auroc_after": fast_auc(y, z[f"p__{STD}"][:, j].astype(float))})
race, auroc = pd.DataFrame(race), pd.DataFrame(auc_rows)
race.to_csv(OUT / "race_after_erasure.csv", index=False, encoding="utf-8-sig")
auroc.to_csv(OUT / "detector_auroc.csv", index=False, encoding="utf-8-sig")
display(race.round(3))
auroc[auroc.label_key.isin([E_COMP, E_AS, M_COMP, M_AS])].assign(change=lambda d: d.auroc_after - d.auroc_before).round(3)

## 8.5 Differences between groups before and after erasure (Figs. 3 and 4, Supplementary Table 7)

Every eraser of a diagnosis is compared with the detector without erasure on the **same bootstrap resamples** (the seeds of notebook 07), so the change and its CI are paired. `change` = after − before.

In [ ]:
def all_predictions(cohort):
    if cohort == "EchoNext":
        base, labels = work("echonext", "detectors"), E_LABELS
        er = lambda fm: np.load(folder / f"erased_{fm}.npz")
        as_neg = lambda fm: np.load(folder / f"erased_as_neg_{fm}.npz")["pred"]
        comp, as_ = E_COMP, E_AS
    else:
        base, labels = work("mimic", "detectors"), M_LABELS
        er = lambda fm: np.load(folder_m / f"erased_{fm}.npz")
        as_neg = lambda fm: np.load(folder_m / f"erased_{fm}.npz")[f"pred__{AS_NEG}"][:, j_as_m]
        comp, as_ = M_COMP, M_AS
    out = {}
    for fm in FMS:
        z, e = np.load(base / f"oof_{fm}.npz"), er(fm)
        out[fm] = {"none": {lab: z["pred"][:, j] for j, lab in enumerate(labels)},
                   STD: {lab: e[f"pred__{STD}"][:, j] for j, lab in enumerate(labels)},
                   SHD_NEG: {comp: e[f"pred__{SHD_NEG}"][:, labels.index(comp)]},
                   AS_NEG: {as_: as_neg(fm)}}
    return out


t0 = time.time()
diff = pd.concat([
    difference_table("EchoNext", E_LABELS, D["Y"], D["g"], strata6(D["age"], D["male"]), all_predictions("EchoNext")),
    difference_table("MIMIC-IV", M_LABELS, Y_lat["MIMIC-IV"], race_index(lat_m.race6), strata6(lat_m.age.to_numpy(float), lat_m.male.to_numpy(int)),
                     all_predictions("MIMIC-IV"))], ignore_index=True)
diff.to_csv(OUT / "differences.csv", index=False, encoding="utf-8-sig")
print(f"bootstrap: {(time.time() - t0) / 60:.1f} min")
q = diff[(diff.pair == "black-white") & (diff.eraser != "none") & diff.measure.isin(["FNR", "FPR", "FLAG", "BA"])
         & diff.label_key.isin([E_COMP, E_AS, M_COMP, M_AS])]
q.assign(before=100 * q.before, after=100 * q.value, change=100 * q.change, lo=100 * q.change_lo, hi=100 * q.change_hi)[
    ["cohort", "label", "eraser", "measure", "fm", "before", "after", "change", "lo", "hi"]].round(1)